# Stage 2 — Instruction Fine-Tuning with LoRA

**Starts from:** the domain-adapted model produced by `01_domain_adaptation_lora.ipynb`
**Data:** [`qiaojin/PubMedQA`](https://huggingface.co/datasets/qiaojin/PubMedQA) `pqa_labeled` —
1,000 expert-annotated question/abstract/answer records
**Hardware:** free-tier Colab T4 · **Runtime:** ~12 minutes

---

## What this notebook does

Notebook 01 left you with a model that writes fluent PubMed abstract prose and **does not answer
questions**. This notebook fixes that by training a second LoRA on paired
instruction/input/output examples.

```
pretrained base
      │
      ├─ stage 1: domain adaptation on raw abstracts   → LoRA #1
      │
      ├─ MERGE LoRA #1 into the base weights           → clean domain-adapted base
      │
      └─ stage 2: instruction tuning on Q/A pairs      → LoRA #2
```

**Why merge instead of stacking two adapters?** Stacked adapters interact in ways that are hard to
predict and harder to validate. Merging produces a single clean base for stage 2, and it means the
stage-2 adapter can be reasoned about on its own.

## This stage has a real metric

Every PubMedQA record carries an expert `final_decision` of **yes / no / maybe**. So instead of
squinting at generated text, we can score **decision accuracy on a held-out set** — and compare it
against two honest baselines:

| baseline | why it matters |
|---|---|
| **Majority class (~55%)** | Always answering "yes". Any model that can't beat this has learned nothing useful. |
| **The base model** | What you get without any instruction tuning at all. |

The articles used here are **disjoint** from the ones notebook 01 trained on — PubMedQA's
`pqa_labeled` and `pqa_unlabeled` configs cover different papers. We assert that rather than
trusting it.

**Set your runtime to a GPU first:** Runtime → Change runtime type → T4 GPU.

## 1. Runtime and dependencies

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "No GPU found — set Runtime > Change runtime type > T4 GPU")

In [ ]:
%pip install -q "transformers==5.15.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.14.0"
# Colab preinstalls torchao 0.10.x. peft's optional torchao integration RAISES rather than
# degrading gracefully when it finds a version below its 0.16.0 minimum, and the check fires
# deep inside get_peft_model(). Nothing here uses torchao, so remove it rather than chase a
# compatible build against Colab's torch.
%pip uninstall -q -y torchao
print("\nRestart the runtime if Colab asks you to, then run this cell again and continue.")

In [ ]:
import gc, json, math, random, re
from pathlib import Path
from collections import Counter

import torch
import transformers
import peft
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No CUDA device. Runtime > Change runtime type > T4 GPU."

# bfloat16 needs Ampere (sm_80) or later; a T4 is Turing (sm_75). Checking compute
# capability directly, because torch.cuda.is_bf16_supported() has returned True on Turing.
SUPPORTS_BF16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
TF_MAJOR = int(transformers.__version__.split(".")[0])
DTYPE_KW = "dtype" if TF_MAJOR >= 5 else "torch_dtype"

MODEL_ID = "unsloth/Llama-3.2-1B"

print(f"transformers {transformers.__version__} | GPU {torch.cuda.get_device_name(0)}")
print(f"compute dtype {DTYPE} | from_pretrained keyword {DTYPE_KW!r}")

# Fail fast on the torchao clash, rather than eight cells from now inside get_peft_model().
import importlib.metadata as _md


def _ver(s):
    return tuple(int(p) for p in s.split("+")[0].split(".")[:3] if p.isdigit())


try:
    _ta = _md.version("torchao")
    if _ver(_ta) < (0, 16, 0):
        raise RuntimeError(
            f"torchao {_ta} is too old for peft {peft.__version__} (needs >= 0.16.0).\n"
            "Nothing in this notebook uses torchao. Fix it with:\n"
            "    !pip uninstall -y torchao\n"
            "then Runtime > Restart session, then Runtime > Run all."
        )
    print(f"torchao      {_ta} (compatible)")
except _md.PackageNotFoundError:
    print("torchao      absent - fine, nothing here uses it")

## 2. Find the stage-1 adapter

Notebook 01 saved it to Drive at `MyDrive/finetuning-demo/stage1-domain-lora`. If you skipped
notebook 01, set `SKIP_STAGE1 = True` and this notebook will instruction-tune the raw base model
instead — a useful ablation, but you lose half the story.

In [ ]:
SKIP_STAGE1 = False

STAGE1_DIR = None
if not SKIP_STAGE1:
    candidates = [
        Path("/content/drive/MyDrive/finetuning-demo/stage1-domain-lora"),
        Path("/content/outputs/stage1-domain-lora"),
    ]
    if not any(c.exists() for c in candidates):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except ImportError:
            pass
    STAGE1_DIR = next((c for c in candidates if (c / "adapter_config.json").exists()), None)

if STAGE1_DIR is None:
    print("No stage-1 adapter found. Either run notebook 01 first, or set SKIP_STAGE1 = True.")
else:
    print(f"Found stage-1 adapter: {STAGE1_DIR}\n")
    cfg = json.loads((STAGE1_DIR / "adapter_config.json").read_text())
    for key in ["base_model_name_or_path", "r", "lora_alpha", "lora_dropout", "target_modules"]:
        print(f"  {key:26s} {cfg.get(key)}")

## 3. Rebuild the domain-adapted base by merging

This is the step to get right. Three sub-steps:

1. Load the **pristine** base model.
2. Attach the stage-1 adapter with `PeftModel.from_pretrained` — *not*
   `AutoModelForCausalLM.from_pretrained(adapter_dir)`, which loads a base model and quietly
   ignores the adapter files sitting next to it.
3. `merge_and_unload()` to fold `(α/r)·BA` into `W`, leaving a plain `LlamaForCausalLM` with no
   PEFT wrapper — a clean base for stage 2.

We snapshot a weight tensor before and after so the merge is **verified**, not assumed.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"          # training, not generation
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD and EOS must stay distinct"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE}).to("cuda")
model.config.pad_token_id = tokenizer.pad_token_id

# Snapshot a weight the stage-1 adapter targeted, so we can prove the merge changed it.
PROBE_WEIGHT = "model.layers.8.mlp.down_proj.weight"
before = dict(model.named_parameters())[PROBE_WEIGHT].detach().clone()
print(f"Loaded pristine {MODEL_ID}")

In [ ]:
if STAGE1_DIR is not None:
    model = PeftModel.from_pretrained(model, STAGE1_DIR)
    print(f"Attached stage-1 adapter ({type(model).__name__})")
    model = model.merge_and_unload()
    print(f"Merged and unwrapped   ({type(model).__name__})")

    after = dict(model.named_parameters())[PROBE_WEIGHT].detach()
    drift = (after.float() - before.float()).abs().max().item()
    changed = not torch.equal(after, before)

    print(f"\nVerification on {PROBE_WEIGHT}")
    print(f"  weights changed : {changed}")
    print(f"  max |delta|     : {drift:.6f}")
    assert changed, (
        "The merge did not change any weights. The adapter was not applied — this is exactly "
        "the failure mode that AutoModelForCausalLM.from_pretrained(adapter_dir) produces."
    )
    print("\n  -> stage-1 knowledge is now inside the base weights")
else:
    print("SKIP_STAGE1 is set: instruction-tuning the raw base model.")

del before
gc.collect(); torch.cuda.empty_cache()
print(f"\nGPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 4. Build the instruction data from PubMedQA

Same source as notebook 01, different config. `pqa_labeled` has 1,000 records that human experts
annotated with a yes/no/maybe decision plus a written justification.

We turn each into an Alpaca-style triple:

- **instruction** — the task, plus the research question
- **input** — the abstract, rendered as `LABEL: text` blocks
- **output** — `Answer: <decision>` then the expert's justification

Putting the decision on the first line is deliberate: it makes the answer **machine-gradable** with
a regex, and it teaches the model to commit before it explains. Structured output like this is a
real production requirement, not a convenience for this notebook.

In [ ]:
from datasets import load_dataset

# --- constants mirrored in scripts/prepare_data.py and notebook 01 ---
DATASET = "qiaojin/PubMedQA"
SEED = 20260815
EVAL_FRACTION = 0.20      # -> 800 train / 200 eval

TASK = (
    "Answer the research question using only the abstract provided. "
    'Begin your reply with "Answer:" followed by yes, no, or maybe, '
    "then justify it in one or two sentences."
)

labeled = load_dataset(DATASET, "pqa_labeled", split="train")
print(labeled)
print("\nexpert decisions across all 1,000 records:",
      dict(Counter(labeled["final_decision"])))

In [ ]:
def join_sections(context) -> str:
    """Render an abstract's sections as `LABEL: text` blocks."""
    labels = context.get("labels") or []
    parts = []
    for i, text in enumerate(context["contexts"]):
        text = " ".join(text.split())
        if not text:
            continue
        label = (labels[i] if i < len(labels) else "").strip()
        parts.append(f"{label}: {text}" if label else text)
    return "\n\n".join(parts)


records = []
for row in labeled:
    abstract = join_sections(row["context"])
    if not abstract:
        continue
    records.append({
        "instruction": f'{TASK}\n\nQuestion: {row["question"].strip()}',
        "input": abstract,
        "output": f'Answer: {row["final_decision"]}\n\n{" ".join(row["long_answer"].split())}',
        "decision": row["final_decision"],
        "pubid": row["pubid"],
    })

print(f"{len(records)} instruction records built")
print("\n--- one rendered record ---")
r = records[0]
print(f"INSTRUCTION:\n{r['instruction']}\n")
print(f"INPUT:\n{r['input'][:400]} ...\n")
print(f"OUTPUT:\n{r['output'][:300]}")

### Stratified split, and a leakage check

`maybe` is only 11% of the data. An unstratified split would leave the eval set with an unstable
number of them and make accuracy noisy, so we split within each decision class.

We also assert that no article appears in both stages. If stage 1 had already read these abstracts,
stage 2's accuracy would be measuring memorisation.

In [ ]:
rng = random.Random(SEED + 1)
by_decision = {}
for r in records:
    by_decision.setdefault(r["decision"], []).append(r)

train_records, eval_records = [], []
for decision in sorted(by_decision):
    rows = sorted(by_decision[decision], key=lambda r: r["pubid"])
    rng.shuffle(rows)
    n_eval = round(len(rows) * EVAL_FRACTION)
    eval_records.extend(rows[:n_eval])
    train_records.extend(rows[n_eval:])
rng.shuffle(train_records); rng.shuffle(eval_records)

print(f"train {len(train_records)}  {dict(Counter(r['decision'] for r in train_records))}")
print(f"eval  {len(eval_records)}  {dict(Counter(r['decision'] for r in eval_records))}")

eval_counts = Counter(r["decision"] for r in eval_records)
MAJORITY_LABEL, majority_n = eval_counts.most_common(1)[0]
MAJORITY_BASELINE = majority_n / len(eval_records)
print(f"\nmajority-class baseline: {MAJORITY_BASELINE:.1%} (always answer '{MAJORITY_LABEL}')")
print("Any model that cannot beat that number has learned nothing useful.")

In [ ]:
# Article-level disjointness between the two stages. pqa_unlabeled (stage 1) and
# pqa_labeled (stage 2) cover different papers, but assert it rather than trust it.
unlabeled_ids = set(load_dataset(DATASET, "pqa_unlabeled", split="train")["pubid"])
stage2_ids = {r["pubid"] for r in records}
overlap = unlabeled_ids & stage2_ids
print(f"pqa_unlabeled articles : {len(unlabeled_ids):,}")
print(f"pqa_labeled articles   : {len(stage2_ids):,}")
print(f"overlap                : {len(overlap)}")
assert not overlap, f"{len(overlap)} articles appear in both stages — stage 2 eval is contaminated"
print("\nNo article is shared between the stages.")

## 5. The prompt template

`unsloth/Llama-3.2-1B` is a **base** model — no chat template ships with it, so we define our own
using the Alpaca format.

**The EOS token is not decoration.** It is the only signal that teaches the model where a response
ends. Leave it off and you get a model that answers correctly and then keeps going — inventing
follow-up questions and answering those. No decoding parameter can repair that.

In [ ]:
PROMPT_WITH_INPUT = (
    "Below is an instruction describing a task, paired with input providing further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

PROMPT_NO_INPUT = (
    "Below is an instruction describing a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n"
)


def build_prompt(record) -> str:
    """The part the model reads but is NOT trained to produce."""
    template = PROMPT_WITH_INPUT if record["input"].strip() else PROMPT_NO_INPUT
    return template.format(instruction=record["instruction"].strip(),
                           input=record["input"].strip())


print(build_prompt(train_records[0])[:700] + " ...")
print("\n>>> TARGET >>>")
print(train_records[0]["output"] + tokenizer.eos_token)

## 6. Completion-only loss masking

The highest-leverage cell in the notebook, and never more so than here.

The naive approach sets `labels = input_ids.copy()`, supervising **every** token — including the
template boilerplate, the research question, and **the entire abstract**. In this dataset the
abstract is ~85% of the sequence. You would be spending the overwhelming majority of the training
budget teaching the model to regurgitate PubMed abstracts, which it neither needs nor benefits
from.

Instead, set `labels = -100` across the prompt span. `-100` is PyTorch's `ignore_index`: those
positions contribute nothing to the loss. The model still *reads* the prompt — it's in
`input_ids` and attention sees it — it just isn't *scored* on it.

```
input_ids :  [Below is ... abstract ... ### Response:\n]  [Answer: yes ...]  [EOS]
labels    :  [-100 -100 -100 -100 ...            -100  ]  [Answer: yes ...]  [EOS]
             └────────── read, not scored ────────────┘   └── supervised ──┘
```

In [ ]:
MAX_LEN = 896   # p99 of this dataset is ~714 tokens; 896 truncates nothing


def tokenize_record(record):
    """Tokenize one record, masking the prompt out of the loss."""
    prompt = build_prompt(record)
    answer = record["output"].strip()

    # add_special_tokens=True puts BOS at the front of the prompt only.
    prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    answer_ids = answer_ids + [tokenizer.eos_token_id]   # <-- teaches the model to stop

    input_ids = (prompt_ids + answer_ids)[:MAX_LEN]
    # -100 over the prompt span, real ids over the answer span.
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]

    return {"input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels}


train_tok = [tokenize_record(r) for r in train_records]
eval_tok = [tokenize_record(r) for r in eval_records]

supervised = sum(sum(1 for t in r["labels"] if t != -100) for r in train_tok)
total = sum(len(r["labels"]) for r in train_tok)
lengths = sorted(len(r["input_ids"]) for r in train_tok)

print(f"sequence length: p50={lengths[len(lengths)//2]} p90={lengths[int(len(lengths)*.9)]} max={lengths[-1]}")
print(f"supervised tokens: {supervised:,} / {total:,} = {supervised/total:.1%}")
print(f"\nWithout the mask, {1 - supervised/total:.0%} of the training signal would go on")
print("reproducing abstracts the model is only supposed to READ.")

truncated = sum(1 for r in train_tok + eval_tok if len(r["input_ids"]) >= MAX_LEN)
assert truncated == 0, f"{truncated} records hit MAX_LEN — a truncated target teaches the model to stop mid-sentence"
print(f"\nrecords truncated at MAX_LEN={MAX_LEN}: {truncated}")

### Sanity check: show exactly which tokens are supervised

Don't take the masking on trust. Decode it.

In [ ]:
sample = train_tok[0]
kept = [t for t in sample["labels"] if t != -100]
masked_n = len(sample["labels"]) - len(kept)

print(f"total tokens  : {len(sample['input_ids'])}")
print(f"masked (-100) : {masked_n}")
print(f"supervised    : {len(kept)}\n")
print("--- MASKED, tail of it (read, not scored) " + "-" * 38)
print(tokenizer.decode(sample["input_ids"][max(0, masked_n - 60):masked_n]))
print("\n--- SUPERVISED (the entire training signal) " + "-" * 36)
print(tokenizer.decode(kept))
print("\nNote the trailing <|end_of_text|>: that is the EOS the model learns to emit.")

## 7. Collator

Sequences here vary a lot in length (p50 ~450, max ~880), so we pad per batch rather than to a
global maximum — considerably less wasted compute.

The important detail: `input_ids` pads with the pad token, `attention_mask` with `0`, and `labels`
with **`-100`**. Padding labels with the pad token id would put the model right back to training
on padding.

In [ ]:
from datasets import Dataset


def collate(features):
    """Dynamic padding. Note the three different pad values."""
    longest = max(len(f["input_ids"]) for f in features)

    def pad(seq, value):
        return seq + [value] * (longest - len(seq))

    return {
        "input_ids": torch.tensor([pad(f["input_ids"], tokenizer.pad_token_id) for f in features]),
        "attention_mask": torch.tensor([pad(f["attention_mask"], 0) for f in features]),
        "labels": torch.tensor([pad(f["labels"], -100) for f in features]),
    }


train_ds = Dataset.from_list(train_tok)
eval_ds = Dataset.from_list(eval_tok)

demo = collate([train_tok[0], train_tok[1]])
print({k: tuple(v.shape) for k, v in demo.items()})
shorter = 0 if len(train_tok[0]["input_ids"]) < len(train_tok[1]["input_ids"]) else 1
print(f"\nlast label of the shorter row: {demo['labels'][shorter][-1].item()}  (-100 if it was padded)")

## 8. Attach a fresh LoRA and train

A **new** adapter, on the merged domain-adapted base. Stage 1's knowledge is in the base weights
now, so this adapter only has to learn one thing: how to respond when addressed.

Batch size 2 with 4 accumulation steps (effective 8). Smaller micro-batches than stage 1 because
sequences are up to 896 tokens rather than a fixed 512.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Same fp16 gotcha as stage 1: trainable params must be fp32 or the grad scaler refuses to
# unscale them. Base weights stay in fp16 — that's where the memory is.
for _, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()

model.config.use_cache = False

In [ ]:
from transformers import Trainer, TrainingArguments

OUT_DIR = "/content/outputs/stage2-instruct-lora"

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not SUPPORTS_BF16,
    bf16=SUPPORTS_BF16,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate,
)

steps = len(train_ds) // (args.per_device_train_batch_size * args.gradient_accumulation_steps)
print(f"optimizer steps: ~{steps * args.num_train_epochs}")
trainer.train()

## 9. Loss curves

These losses are **not comparable to notebook 01's**. Stage 1 scored every token in a packed
block; stage 2 scores only the ~13% of tokens that are response. Different denominators,
different tasks.

In [ ]:
history = trainer.state.log_history
train_pts = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_pts = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(9, 4))
plt.plot(*zip(*train_pts), label="train loss", color="#4C72B0", alpha=0.8)
if eval_pts:
    plt.plot(*zip(*eval_pts), label="eval loss (held-out records)", color="#C44E52", marker="o")
plt.xlabel("epoch"); plt.ylabel("loss (response tokens only)"); plt.legend()
plt.title("Stage 2: instruction tuning on PubMedQA"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

if eval_pts:
    print(f"eval loss: {eval_pts[0][1]:.3f} (epoch {eval_pts[0][0]:.0f}) "
          f"-> {eval_pts[-1][1]:.3f} (epoch {eval_pts[-1][0]:.0f})")

## 10. Save the stage-2 adapter

Saving **before** the evaluation below, because that section frees the model to make room for
loading the pristine base.

In [ ]:
import shutil

ADAPTER_DIR = Path(OUT_DIR)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Saved to {ADAPTER_DIR}  ({size_mb:.1f} MB)")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/finetuning-demo/stage2-instruct-lora")
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(ADAPTER_DIR, dest)
    print(f"Copied to {dest}")
except ImportError:
    print("Not running in Colab — adapter is on local disk only.")

## 11. The metric: decision accuracy on held-out records

Now the part that makes this stage measurable. For each held-out record we generate a response and
parse the leading `Answer: <yes|no|maybe>`, then compare against the expert label.

Three systems, scored identically:

| | |
|---|---|
| **base** | `unsloth/Llama-3.2-1B`, untouched |
| **+ domain** | after stage 1 — the merged weights under the stage-2 adapter |
| **+ domain + instruct** | after stage 2 — the full pipeline |

**Memory trick:** the middle system comes free. `model.disable_adapter()` switches the stage-2
LoRA off in place, and what's left is precisely the merged stage-1 model. Only the pristine base
needs a separate load.

A model that cannot produce a parseable `Answer:` line at all is scored as wrong — that is the
honest treatment, since an unparseable answer is useless downstream. We report the parse rate
separately so you can see *why* a system scored badly.

In [ ]:
DECISION_RE = re.compile(r"answer\s*:?\s*\b(yes|no|maybe)\b", re.IGNORECASE)


@torch.no_grad()
def generate_batch(m, batch, max_new_tokens=120):
    """Greedy, left-padded batched generation. Deterministic so runs are comparable."""
    m.eval()
    m.config.use_cache = True
    prompts = [build_prompt(r) for r in batch]
    # Left padding for generation: every sequence then ends at the same position,
    # which is where the model continues from. (Training used right padding.)
    tokenizer.padding_side = "left"
    enc = tokenizer(prompts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_LEN).to("cuda")
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                     eos_token_id=tokenizer.eos_token_id,
                     pad_token_id=tokenizer.pad_token_id)
    tokenizer.padding_side = "right"
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]


def score(m, dataset, label, batch_size=8):
    """Decision accuracy + parse rate over the held-out set."""
    texts = []
    for i in range(0, len(dataset), batch_size):
        texts.extend(generate_batch(m, dataset[i:i + batch_size]))
        print(f"\r  {label}: {min(i + batch_size, len(dataset))}/{len(dataset)}", end="")
    print()
    preds = []
    for t in texts:
        hit = DECISION_RE.search(t)
        preds.append(hit.group(1).lower() if hit else None)
    correct = sum(1 for p, r in zip(preds, dataset) if p == r["decision"])
    parsed = sum(1 for p in preds if p is not None)
    return {"accuracy": correct / len(dataset), "parse_rate": parsed / len(dataset),
            "preds": preds, "texts": texts}

In [ ]:
# Scored on the full 200-record held-out set. Takes a few minutes.
results = {}
results["+ domain + instruct"] = score(model, eval_records, "stage 2")
with model.disable_adapter():
    results["+ domain"] = score(model, eval_records, "stage 1")
print("\ndone with the adapter-based systems")

In [ ]:
# Free everything, then load the pristine base for the third row.
del trainer
gc.collect(); torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: DTYPE}).to("cuda")
base_model.config.pad_token_id = tokenizer.pad_token_id
results["base"] = score(base_model, eval_records, "base")

del base_model
gc.collect(); torch.cuda.empty_cache()
print("\ndone")

In [ ]:
gold = [r["decision"] for r in eval_records]

print(f"Held-out set: {len(eval_records)} records   {dict(Counter(gold))}\n")
print(f"{'system':26s}{'accuracy':>10s}{'parse rate':>13s}")
print("-" * 49)
print(f"{'majority class (' + MAJORITY_LABEL + ')':26s}{MAJORITY_BASELINE:>9.1%}{'n/a':>13s}")
for name in ["base", "+ domain", "+ domain + instruct"]:
    r = results[name]
    print(f"{name:26s}{r['accuracy']:>9.1%}{r['parse_rate']:>12.1%}")
print("-" * 49)

final = results["+ domain + instruct"]["accuracy"]
print(f"\nvs majority class : {final - MAJORITY_BASELINE:+.1%}")
print(f"vs base model     : {final - results['base']['accuracy']:+.1%}")

### Read this carefully

- The **base** and **+ domain** rows will usually have a low *parse rate*. Neither has ever been
  taught to emit `Answer: yes`; they continue the prompt instead. Their accuracy is low mostly
  because they don't answer in a usable form at all — which is precisely the capability stage 2
  installs.
- If **+ domain + instruct** lands near 55%, check the confusion matrix below. Collapsing onto
  "yes" for everything is the most common outcome at this data scale, and it means the model
  learned the *format* but not the *task*.
- Beating majority class by a clear margin on 800 training examples with a 1B model would be a
  strong result. PubMedQA is hard: published fine-tuned 7B–70B models sit in the 55–78% range on
  the official test split.

In [ ]:
# Confusion matrix for the final model — is it actually discriminating, or just saying "yes"?
LABELS = ["yes", "no", "maybe"]
preds = results["+ domain + instruct"]["preds"]

print("rows = expert label, cols = prediction\n")
print(f"{'':8s}" + "".join(f"{c:>8s}" for c in LABELS) + f"{'unparsed':>10s}")
for actual in LABELS:
    row = [sum(1 for p, g in zip(preds, gold) if g == actual and p == c) for c in LABELS]
    unparsed = sum(1 for p, g in zip(preds, gold) if g == actual and p is None)
    print(f"{actual:8s}" + "".join(f"{v:>8d}" for v in row) + f"{unparsed:>10d}")

print(f"\nprediction distribution: {dict(Counter(p or 'unparsed' for p in preds))}")
print(f"expert distribution    : {dict(Counter(gold))}")

## 12. Qualitative side-by-side

The numbers say what changed. These show it.

In [ ]:
def show(text, limit=420):
    text = " ".join(text.split())
    return text if len(text) <= limit else text[:limit] + " ..."


for i in range(4):
    r = eval_records[i]
    print("=" * 112)
    print(f"Q: {r['instruction'].split('Question: ')[-1]}")
    print(f"   [abstract] {show(r['input'], 180)}\n")
    print(f"  [1] base                : {show(results['base']['texts'][i])}\n")
    print(f"  [2] + domain            : {show(results['+ domain']['texts'][i])}\n")
    print(f"  [3] + domain + instruct : {show(results['+ domain + instruct']['texts'][i])}\n")
    print(f"  EXPERT                  : {show(r['output'])}")
print("=" * 112)

In [ ]:
# How long does each system run before it stops? Stage 2 should stop on its own.
print("base   st1   st2   (tokens generated before EOS or the 120-token cap)")
for i in range(min(8, len(eval_records))):
    lens = [len(tokenizer(results[k]["texts"][i], add_special_tokens=False)["input_ids"])
            for k in ("base", "+ domain", "+ domain + instruct")]
    print(f"{lens[0]:>4}  {lens[1]:>4}  {lens[2]:>4}")
avg = {k: sum(len(tokenizer(t, add_special_tokens=False)["input_ids"])
              for t in results[k]["texts"]) / len(eval_records)
       for k in ("base", "+ domain", "+ domain + instruct")}
print(f"\nmean generated length: base {avg['base']:.0f}, "
      f"+domain {avg['+ domain']:.0f}, +instruct {avg['+ domain + instruct']:.0f}")
print("A stage-2 mean well under 120 means the model learned to stop on its own.")

## 13. What you built

```
unsloth/Llama-3.2-1B  (base, no instruction tuning)
    └─ stage 1: LoRA on ~290k tokens of PubMed abstracts   ~11M params
         └─ merged into base weights
              └─ stage 2: LoRA on 800 PubMedQA Q/A pairs   ~11M params
```

Two adapters, ~45 MB each, on a 1.24B-parameter model, trained end to end on a free T4 — scored
against an objective metric on articles neither stage trained on.

### The four bugs this pipeline avoids

1. Loading an adapter with `AutoModelForCausalLM.from_pretrained(adapter_dir)` — silently gives
   you the base model. We used `PeftModel.from_pretrained` and asserted the weights changed.
2. `labels = input_ids.copy()` with `padding="max_length"` — trains on padding. We packed in
   stage 1 and masked with `-100` in stage 2.
3. Supervising the prompt — here that would mean spending ~87% of the budget regenerating
   abstracts. We masked the prompt span.
4. No EOS on targets — the model never stops. We appended it and measured generation lengths.

### Honest limitations

- **800 training examples is very small**, and 1B is a small model for a task that rewards
  reasoning over a dense abstract. Expect correct *form* more reliably than correct *content*.
- **Our split is not the official PubMedQA benchmark split** (which is 450 train / 50 dev /
  500 test). Do not compare these numbers to published leaderboard results.
- **`maybe` is only 11% of the data** and is by far the hardest class. Check the confusion matrix
  rather than the headline accuracy.
- **Nothing here is medical advice.** A 1B model fine-tuned on 800 abstracts is a training
  exercise, not a clinical tool.

### Where to go next

| | |
|---|---|
| **More data** | `pqa_artificial` has 211k auto-labelled records — the single highest-return change. |
| **TRL `SFTTrainer`** | Does the masking and packing for you via `DataCollatorForCompletionOnlyLM`. Worth using once you understand what it's doing. |
| **Classification head** | For yes/no/maybe alone, a sequence-classification head beats generation and is far cheaper to score. |
| **QLoRA** | 4-bit base weights via `bitsandbytes` — needed once the model is 7B+. **Notebook 04 does this on an 8B base**, structured so it never has to merge, and measures what merging into 4-bit would have cost. |
| **Unsloth** | ~2x faster with lower memory for exactly this workload. |
| **Preference tuning** | DPO / ORPO on preference pairs — the stage after this one. |

See `docs/concepts.md` for the reasoning behind each decision here.